# Chapter 10: Binary Search

The programs from Chapter 10 of *Data Structures and Algorithms in Python, Hands-On* by Aadhavan Muthurengan, in the book's order. Each code cell is one program; the same program is in this folder as a `.py` file, and its expected output is in the `.out` file with the same name. Benchmark timings differ on every computer and every run.

Run the cells from top to bottom (**Run > Run All Cells**). The programs import each other's files, so keep this notebook in its chapter folder.

## The Algorithm and Its Invariant

`binary_search.py`

In [1]:
"""Binary search in a sorted sequence, and its two useful cousins."""


def binary_search(items, target):
    """Index of target in sorted items, or -1.

    Invariant: if target is anywhere, it is in items[lo:hi]. Each step
    halves that range, so it takes about log2(n) steps.
    """
    lo, hi = 0, len(items)
    while lo < hi:
        mid = (lo + hi) // 2
        if items[mid] == target:
            return mid
        if items[mid] < target:
            lo = mid + 1  # target can only be to the right of mid
        else:
            hi = mid  # target can only be to the left of mid
    return -1


def lower_bound(items, target):
    """The first index whose item is >= target (len(items) if none).
    It is where target would be inserted to keep the order."""
    lo, hi = 0, len(items)
    while lo < hi:
        mid = (lo + hi) // 2
        if items[mid] < target:
            lo = mid + 1
        else:
            hi = mid
    return lo


def upper_bound(items, target):
    """The first index whose item is > target (len(items) if none)."""
    lo, hi = 0, len(items)
    while lo < hi:
        mid = (lo + hi) // 2
        if items[mid] <= target:
            lo = mid + 1
        else:
            hi = mid
    return lo

`count_steps.py`

In [2]:
"""How many steps does binary search take? range() can be indexed
like a list without storing it, so we can search a billion numbers."""


def steps_to_find(items, target):
    lo, hi, steps = 0, len(items), 0
    while lo < hi:
        steps += 1
        mid = (lo + hi) // 2
        if items[mid] == target:
            break
        if items[mid] < target:
            lo = mid + 1
        else:
            hi = mid
    return steps


for n in [1_000, 1_000_000, 1_000_000_000]:
    worst = max(steps_to_find(range(n), t) for t in (-1, 0, n - 1, n))
    print(f"{n:>13,} items: at most {worst} steps")

        1,000 items: at most 10 steps
    1,000,000 items: at most 20 steps
1,000,000,000 items: at most 30 steps


## The bisect Module

`bisect_demo.py`

In [3]:
"""The bisect module: binary search in the standard library."""

import bisect

# Grades: the score where each grade starts, and the grade names
cutoffs = [60, 70, 80, 90]
grades = "FDCBA"
for score in [55, 60, 79, 90, 100]:
    print(score, grades[bisect.bisect_right(cutoffs, score)])

# Keep a list sorted as items arrive
scores = []
for s in [70, 40, 90, 55]:
    bisect.insort(scores, s)
print(scores)

55 F
60 D
79 C
90 A
100 A
[40, 55, 70, 90]


## Binary Search on the Answer

`search_answer.py`

In [4]:
"""Binary search on the answer: when you can check a guess quickly and
every guess above the answer passes, search the range of answers.

Problem: packages with these weights must ship in order within `days`
days. What is the smallest daily capacity that works?
"""


def days_needed(weights, capacity):
    days, load = 1, 0
    for w in weights:
        if load + w > capacity:  # this package starts a new day
            days, load = days + 1, 0
        load += w
    return days


def smallest_capacity(weights, days):
    lo, hi = max(weights), sum(weights)  # the answer is in lo..hi
    while lo < hi:
        mid = (lo + hi) // 2
        if days_needed(weights, mid) <= days:
            hi = mid  # mid works: maybe something smaller does too
        else:
            lo = mid + 1  # mid fails: the answer is bigger
    return lo


weights = [3, 2, 2, 4, 1, 4, 5, 6, 2, 3]
for days in [1, 3, 5, 10]:
    print(f"{days:>2} days: capacity {smallest_capacity(weights, days)}")

 1 days: capacity 32
 3 days: capacity 11
 5 days: capacity 9
10 days: capacity 6


## Timing

`bench_search.py`

In [5]:
"""Searching a sorted list: scanning, our binary search, and bisect."""

import bisect

import bench
from binary_search import binary_search


def make(n):
    return list(range(n)), n - 1  # the last item: the scan's worst case


def scan(items, target):
    return target in items


def with_bisect(items, target):
    i = bisect.bisect_left(items, target)
    return i < len(items) and items[i] == target


sizes = [1_000, 10_000, 100_000, 1_000_000]
results = {
    "scan (in)": bench.doubling(scan, make, sizes),
    "binary_search": bench.doubling(binary_search, make, sizes),
    "bisect": bench.doubling(with_bisect, make, sizes),
}
bench.table(results)
bench.chart(results, "ch10_search.png", "Searching a sorted list", log=True)

         n       scan (in)   binary_search          bisect
     1,000         6.41 us          609 ns          173 ns
    10,000   65.3 us x10.2     906 ns x1.5     182 ns x1.1
   100,000    703 us x10.8    1.03 us x1.1     220 ns x1.2
 1,000,000     11 ms x15.7    1.24 us x1.2     243 ns x1.1
